## 1. Shor Algorithm

Designed to solve the problem of integer factorization—finding the prime numbers that, when multiplied together, produce a specific large number

##### Classical Computing

Classical algorithms like the General Number Field Sieve are exponential. While the exact formula is complex, it scales roughly like:

$$T_{classical} \approx e^{1.9 \cdot n^{1/3} (\log n)^{2/3}}$$

For a 2048-bit number, $n$ is so large that the number of operations exceeds $2^{100}$.The Result: Even if you used every computer on Earth, it would take billions of years to finish. If you add just 10 bits to the key, the difficulty nearly doubles.

##### Quantum Computing: Polynomial Growth

Shor’s Algorithm is polynomial, scaling approximately as:$$T_{quantum} \approx n^3$$

For a 2048-bit number, the "work" required is roughly $2048^3$ (about 8.5 billion operations). While 8.5 billion sounds like a lot, quantum gates operate so fast that a stable quantum computer could finish the task in hours or days.

1. Pick a random ($a$) with $(1 < a < N)$. If $(\gcd(a, N) \neq 1)$, you already found a factor.

2. Use QPE to find the order ($r$) of $(a \mod N)$: $(a^r \equiv 1 \pmod N)$.

3. If ($r$) is even and $(a^{r/2} \not\equiv -1 \pmod N)$, compute: $\gcd(a^{r/2}-1,, N), \quad \gcd(a^{r/2}+1,, N)$. These give non‑trivial factors.

4. If conditions fail, pick a new ($a$) and repeat.

### 1.2 Quantum Phase Estimation

To understand Quantum Phase Estimation (QPE), we represent the phase as a value $\theta$ in the eigenvalue equation for a unitary operator $U$:

$$U|\psi\rangle = e^{2\pi i \theta}|\psi\rangle$$

Here, $|\psi\rangle$ is the eigenvector and $e^{2\pi i \theta}$ is the eigenvalue. The goal of QPE is to estimate the value of $\theta$ (where $0 \leq \theta < 1$).

##### The Mathematical Description

1. **State Preparation**: We start with $n$ counting qubits in the $|0\rangle$ state and a target register in the state $|\psi\rangle$. Applying Hadamard gates ($H^{\otimes n}$) to the counting qubits creates a uniform superposition:$$\frac{1}{2^{n/2}} \sum_{j=0}^{2^n-1} |j\rangle |\psi\rangle$$

2. **Controlled-U Operations (Phase Kickback)**: We apply $U$ controlled by the counting qubits, but raised to successive powers of 2. Because of Phase Kickback, the phase is "kicked up" into the counting register:$$\frac{1}{2^{n/2}} \sum_{j=0}^{2^n-1} e^{2\pi i j \theta} |j\rangle |\psi\rangle$$

3. **The Inverse QFT**: The state of the counting qubits now looks like a Fourier-transformed version of the phase. To extract $\theta$, we apply the Inverse Quantum Fourier Transform ($QFT^\dagger$):$$QFT^\dagger \left( \frac{1}{2^{n/2}} \sum_{j=0}^{2^n-1} e^{2\pi i j \theta} |j\rangle \right) = |\tilde{\theta}\rangle$$

4. **Measurement**: Measuring the register gives a bitstring $x$. The estimate for the phase is:$$\theta \approx \frac{x}{2^n}$$

##### Summary of the variables

- $n$: The number of counting qubits (determines the precision). 
- $U$: The unitary operator whose phase we want to find.
- $\theta$: The unknown phase value we are "estimating."

In [8]:
import math
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFT
from qiskit_aer import AerSimulator

class ShorAlgorithm:
    def __init__(self, N, a):
        self.N = N
        self.a = a
        # For N=15, we need 8 counting qubits for precision 
        # and 4 qubits to hold the number 15 (2^4 = 16)
        self.n_count = 8 
        self.n_target = 4
        self.qc = QuantumCircuit(self.n_count + self.n_target, self.n_count)

    def modular_exponentiation(self):
        """Adds the controlled modular multiplication gates to the circuit."""
        # Start the target register in state |1>
        self.qc.x(self.n_count) 
        
        # Apply controlled-U gates for Phase Estimation
        for q in range(self.n_count):
            # We apply a^(2^q) mod N
            # For N=15, we use a specialized gate (see helper below)
            self.qc.append(self.get_amod15(self.a, 2**q), 
                           [q] + [i + self.n_count for i in range(self.n_target)])

    def get_amod15(self, a, power):
        """Helper to create the U gate for a^power mod 15."""
        U = QuantumCircuit(4)        
        for _ in range(power):
            if a in [2, 13]:
                U.swap(0, 1); U.swap(1, 2); U.swap(2, 3)
            if a in [7, 8]:
                U.swap(2, 3); U.swap(1, 2); U.swap(0, 1)
            if a == 11:
                U.swap(0, 2); U.swap(1, 3)
            if a in [13, 11, 8]:
                for j in range(4): U.x(j)
        return U.to_gate(label=f"{a}^{power} mod {self.N}").control()

    def compute(self):
        # 1. Superposition
        self.qc.h(range(self.n_count))
        
        # 2. Modular Exponentiation
        self.modular_exponentiation()
        
        # 3. Inverse QFT (The core of Phase Estimation)
        self.qc.append(QFT(self.n_count, inverse=True).to_gate(), range(self.n_count))
        
        # 4. Measure
        self.qc.measure(range(self.n_count), range(self.n_count))
        
        # 5. Simulate
        backend = AerSimulator()
        t_qc = transpile(self.qc, backend)
        return backend.run(t_qc).result().get_counts()

# --- Execution ---
shor = ShorAlgorithm(N=15, a=7)
counts = shor.compute()
print("Measured Phases:", counts)

Measured Phases: {'00000000': 251, '01000000': 270, '11000000': 243, '10000000': 260}


/var/folders/td/rtvtvvjn3x1gfqswnc7bb6hm0000gn/T/ipykernel_16172/807974054.py:51: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  self.qc.append(QFT(self.n_count, inverse=True).to_gate(), range(self.n_count))
